## 0_MaxResAndStack.py

In [1]:
import pyrealsense2 as rs
import numpy as np
import cv2

# 1. สร้าง Pipeline และ Config
pipeline = rs.pipeline()
config = rs.config()

# ---------------------------------------------------------
# ตั้งค่าความละเอียดสูงสุดของกล้อง D415
# ---------------------------------------------------------
# RGB: 1920 x 1080 ที่ 30 FPS
config.enable_stream(rs.stream.color, 1920, 1080, rs.format.bgr8, 30)
# Depth: 1280 x 720 ที่ 30 FPS
config.enable_stream(rs.stream.depth, 1280, 720, rs.format.z16, 30)

print("กำลังเชื่อมต่อกล้อง Intel RealSense ด้วยความละเอียดสูงสุด...")
print("⚠️ (ตรวจสอบให้แน่ใจว่าเสียบสายเข้ากับพอร์ต USB 3.0)")

# 2. เริ่มต้นการเชื่อมต่อ
profile = pipeline.start(config)

# 3. ตั้งค่าเปิด IR Projector
depth_sensor = profile.get_device().first_depth_sensor()
if depth_sensor.supports(rs.option.emitter_enabled):
    depth_sensor.set_option(rs.option.emitter_enabled, 1) 

# ตัวช่วยสำหรับใส่สีให้ภาพ Depth
colorizer = rs.colorizer()

print("กำลังสตรีมภาพ...")
print("⚠️ คลิกที่หน้าต่างวิดีโอ แล้วกดปุ่ม 'q' หรือ 'ESC' เพื่อปิดโปรแกรม")

try:
    while True:
        # รอรับชุดข้อมูลเฟรมล่าสุดจากกล้อง
        frames = pipeline.wait_for_frames()

        # ดึงเฟรมแต่ละประเภท
        color_frame = frames.get_color_frame()
        depth_frame = frames.get_depth_frame()

        if not color_frame or not depth_frame:
            continue

        # ---------------------------------------------------------
        # แปลงข้อมูลเฟรมเป็น Numpy Array
        # ---------------------------------------------------------
        # ภาพ RGB (ขนาดดั้งเดิม 1920 x 1080)
        color_image = np.asanyarray(color_frame.get_data())
        
        # ภาพ Depth (ขนาดดั้งเดิม 1280 x 720)
        depth_color_image = np.asanyarray(colorizer.colorize(depth_frame).get_data())

        # ---------------------------------------------------------
        # ใส่ข้อความกำกับ (Labels)
        # ---------------------------------------------------------
        font = cv2.FONT_HERSHEY_SIMPLEX
        color_label = (0, 255, 0)
        thickness = 3 # เพิ่มความหนาของตัวอักษรเพราะภาพต้นฉบับใหญ่ขึ้นมาก
        
        cv2.putText(color_image, "RGB (1920x1080)", (20, 50), font, 1.5, color_label, thickness)
        cv2.putText(depth_color_image, "Depth (1280x720)", (20, 50), font, 1.5, color_label, thickness)

        # ---------------------------------------------------------
        # ปรับขนาดภาพให้เท่ากันเพื่อแสดงผล (Display Only)
        # ---------------------------------------------------------
        # ขยายภาพ Depth จาก 720p เป็น 1080p เพื่อให้ความสูงเท่ากับภาพ RGB (สัดส่วน 16:9 เท่ากัน)
        depth_color_resized = cv2.resize(depth_color_image, (color_image.shape[1], color_image.shape[0]))

        # นำภาพมาต่อกันแนวนอน (ซ้าย: RGB | ขวา: Depth)
        images_hstack = np.hstack((color_image, depth_color_resized))

        # ---------------------------------------------------------
        # ย่อขนาดสำหรับแสดงบนหน้าจอ (Window Scaling)
        # ---------------------------------------------------------
        # ภาพต่อกันแนวนอนตอนนี้มีขนาด 3840 x 1080 พิกเซล ซึ่งใหญ่กว่าจอคอมพิวเตอร์ทั่วไป
        # เราจึงต้องย่อ "เฉพาะตอนแสดงหน้าต่าง" ลง 50% ให้เหลือ 1920 x 540
        # (ข้อมูลดิบในตัวแปร color_frame และ depth_frame ยังคงเป็นความละเอียดสูงสุด)
        display_image = cv2.resize(images_hstack, (0, 0), fx=0.5, fy=0.5)

        # แสดงหน้าต่าง OpenCV
        cv2.imshow('Intel RealSense - Max Resolution', display_image)

        # ตรวจสอบการกดปุ่ม 'q' หรือ 'ESC' (27) เพื่อออกจากลูป
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:
            break

finally:
    # คืนทรัพยากรกล้อง
    pipeline.stop()
    cv2.destroyAllWindows()
    cv2.waitKey(1) 
    print("ปิดการสตรีมและเคลียร์หน้าต่างเรียบร้อยแล้ว")

กำลังเชื่อมต่อกล้อง Intel RealSense ด้วยความละเอียดสูงสุด...
⚠️ (ตรวจสอบให้แน่ใจว่าเสียบสายเข้ากับพอร์ต USB 3.0)
กำลังสตรีมภาพ...
⚠️ คลิกที่หน้าต่างวิดีโอ แล้วกดปุ่ม 'q' หรือ 'ESC' เพื่อปิดโปรแกรม
ปิดการสตรีมและเคลียร์หน้าต่างเรียบร้อยแล้ว


# 1_RawStream

In [ ]:
import pyrealsense2 as rs
import numpy as np
import cv2

# 1. สร้าง Pipeline และ Config
pipeline = rs.pipeline()
config = rs.config()

# ---------------------------------------------------------
# ตั้งค่าความละเอียดสูงสุดของกล้อง D415
# ---------------------------------------------------------
# RGB: 1920 x 1080 ที่ 30 FPS
config.enable_stream(rs.stream.color, 1920, 1080, rs.format.bgr8, 30)
# Depth: 1280 x 720 ที่ 30 FPS
config.enable_stream(rs.stream.depth, 1280, 720, rs.format.z16, 30)

print("กำลังเชื่อมต่อกล้อง Intel RealSense ด้วยความละเอียดสูงสุด...")
print("⚠️ (ตรวจสอบให้แน่ใจว่าเสียบสายเข้ากับพอร์ต USB 3.0)")

# 2. เริ่มต้นการเชื่อมต่อ
profile = pipeline.start(config)

# 3. ตั้งค่าเปิด IR Projector
depth_sensor = profile.get_device().first_depth_sensor()
if depth_sensor.supports(rs.option.emitter_enabled):
    depth_sensor.set_option(rs.option.emitter_enabled, 1) 

# ตัวช่วยสำหรับใส่สีให้ภาพ Depth
colorizer = rs.colorizer()

print("กำลังสตรีมภาพ...")
print("⚠️ คลิกที่หน้าต่างวิดีโอ แล้วกดปุ่ม 'q' หรือ 'ESC' เพื่อปิดโปรแกรม")

try:
    while True:
        # รอรับชุดข้อมูลเฟรมล่าสุดจากกล้อง
        frames = pipeline.wait_for_frames()

        # ดึงเฟรมแต่ละประเภท
        color_frame = frames.get_color_frame()
        depth_frame = frames.get_depth_frame()

        if not color_frame or not depth_frame:
            continue

        # ---------------------------------------------------------
        # แปลงข้อมูลเฟรมเป็น Numpy Array
        # ---------------------------------------------------------
        # ภาพ RGB (ขนาดดั้งเดิม 1920 x 1080)
        color_image = np.asanyarray(color_frame.get_data())
        
        # ภาพ Depth (ขนาดดั้งเดิม 1280 x 720)
        depth_color_image = np.asanyarray(colorizer.colorize(depth_frame).get_data())

        # ---------------------------------------------------------
        # ใส่ข้อความกำกับ (Labels)
        # ---------------------------------------------------------
        font = cv2.FONT_HERSHEY_SIMPLEX
        color_label = (0, 255, 0)
        thickness = 3 # เพิ่มความหนาของตัวอักษรเพราะภาพต้นฉบับใหญ่ขึ้นมาก
        
        cv2.putText(color_image, "RGB (1920x1080)", (20, 50), font, 1.5, color_label, thickness)
        cv2.putText(depth_color_image, "Depth (1280x720)", (20, 50), font, 1.5, color_label, thickness)

        # ---------------------------------------------------------
        # ปรับขนาดภาพให้เท่ากันเพื่อแสดงผล (Display Only)
        # ---------------------------------------------------------
        # ขยายภาพ Depth จาก 720p เป็น 1080p เพื่อให้ความสูงเท่ากับภาพ RGB (สัดส่วน 16:9 เท่ากัน)
        depth_color_resized = cv2.resize(depth_color_image, (color_image.shape[1], color_image.shape[0]))

        # นำภาพมาต่อกันแนวนอน (ซ้าย: RGB | ขวา: Depth)
        images_hstack = np.hstack((color_image, depth_color_resized))

        # ---------------------------------------------------------
        # ย่อขนาดสำหรับแสดงบนหน้าจอ (Window Scaling)
        # ---------------------------------------------------------
        # ภาพต่อกันแนวนอนตอนนี้มีขนาด 3840 x 1080 พิกเซล ซึ่งใหญ่กว่าจอคอมพิวเตอร์ทั่วไป
        # เราจึงต้องย่อ "เฉพาะตอนแสดงหน้าต่าง" ลง 50% ให้เหลือ 1920 x 540
        # (ข้อมูลดิบในตัวแปร color_frame และ depth_frame ยังคงเป็นความละเอียดสูงสุด)
        display_image = cv2.resize(images_hstack, (0, 0), fx=0.5, fy=0.5)

        # แสดงหน้าต่าง OpenCV
        cv2.imshow('Intel RealSense - Max Resolution', display_image)

        # ตรวจสอบการกดปุ่ม 'q' หรือ 'ESC' (27) เพื่อออกจากลูป
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:
            break

finally:
    # คืนทรัพยากรกล้อง
    pipeline.stop()
    cv2.destroyAllWindows()
    cv2.waitKey(1) 
    print("ปิดการสตรีมและเคลียร์หน้าต่างเรียบร้อยแล้ว")

# 2_PostProcessFilter

In [ ]:
import pyrealsense2 as rs
import numpy as np
import cv2

# 1. สร้าง Pipeline และ Config
pipeline = rs.pipeline()
config = rs.config()

# ---------------------------------------------------------
# ตั้งค่าความละเอียดสูงสุดของกล้อง D415
# ---------------------------------------------------------
# RGB: 1920 x 1080 ที่ 30 FPS
config.enable_stream(rs.stream.color, 1920, 1080, rs.format.bgr8, 30)
# Depth: 1280 x 720 ที่ 30 FPS
config.enable_stream(rs.stream.depth, 1280, 720, rs.format.z16, 30)

print("กำลังเชื่อมต่อกล้อง Intel RealSense...")
print("⚠️ (ตรวจสอบให้แน่ใจว่าเสียบสายเข้ากับพอร์ต USB 3.0)")

# 2. เริ่มต้นการเชื่อมต่อ
profile = pipeline.start(config)

# 3. ตั้งค่าเปิด IR Projector
depth_sensor = profile.get_device().first_depth_sensor()
if depth_sensor.supports(rs.option.emitter_enabled):
    depth_sensor.set_option(rs.option.emitter_enabled, 1) 

# ---------------------------------------------------------
# 4. ประกาศตัวแปรสำหรับ Post-Processing Filters
# ---------------------------------------------------------
spatial_filter = rs.spatial_filter()
temporal_filter = rs.temporal_filter()
hole_filling = rs.hole_filling_filter()

# ตัวช่วยสำหรับใส่สีให้ภาพ Depth
colorizer = rs.colorizer()

print("กำลังสตรีมภาพความละเอียดสูงสุด พร้อมเปิดใช้งาน Filters...")
print("⚠️ คลิกที่หน้าต่างวิดีโอ แล้วกดปุ่ม 'q' หรือ 'ESC' เพื่อปิดโปรแกรม")

try:
    while True:
        # รอรับชุดข้อมูลเฟรมล่าสุดจากกล้อง
        frames = pipeline.wait_for_frames()

        # ดึงเฟรมแต่ละประเภท
        color_frame = frames.get_color_frame()
        depth_frame = frames.get_depth_frame()

        if not color_frame or not depth_frame:
            continue

        # ---------------------------------------------------------
        # 5. นำภาพ Depth ผ่านตัวกรอง (Apply Filters)
        # ---------------------------------------------------------
        # เรียงลำดับ: Spatial (เกลี่ยพิกเซล) -> Temporal (ลดกระพริบ) -> Hole Filling (ถมดำ)
        filtered_depth = spatial_filter.process(depth_frame)
        filtered_depth = temporal_filter.process(filtered_depth)
        filtered_depth = hole_filling.process(filtered_depth)

        # ---------------------------------------------------------
        # แปลงข้อมูลเฟรมเป็น Numpy Array
        # ---------------------------------------------------------
        # ภาพ RGB (ขนาด 1920 x 1080)
        color_image = np.asanyarray(color_frame.get_data())
        
        # ภาพ Depth ที่ผ่านการทำ Filter แล้ว (ขนาด 1280 x 720)
        depth_color_image = np.asanyarray(colorizer.colorize(filtered_depth).get_data())

        # ---------------------------------------------------------
        # ใส่ข้อความกำกับ (Labels)
        # ---------------------------------------------------------
        font = cv2.FONT_HERSHEY_SIMPLEX
        color_label = (0, 255, 0)
        thickness = 3 
        
        cv2.putText(color_image, "RGB (1920x1080)", (20, 50), font, 1.5, color_label, thickness)
        cv2.putText(depth_color_image, "Filtered Depth (1280x720)", (20, 50), font, 1.5, color_label, thickness)

        # ---------------------------------------------------------
        # ปรับขนาดและจัด Layout
        # ---------------------------------------------------------
        # ขยายภาพ Depth จาก 720p เป็น 1080p เพื่อให้ความสูงเท่ากับภาพ RGB
        depth_color_resized = cv2.resize(depth_color_image, (color_image.shape[1], color_image.shape[0]))

        # นำภาพมาต่อกันแนวนอน (ซ้าย: RGB | ขวา: Filtered Depth)
        images_hstack = np.hstack((color_image, depth_color_resized))

        # ย่อขนาดลง 50% ให้เหลือ 1920 x 540 เพื่อให้แสดงผลบนหน้าจอคอมพิวเตอร์ได้พอดี
        display_image = cv2.resize(images_hstack, (0, 0), fx=0.5, fy=0.5)

        # แสดงหน้าต่าง OpenCV
        cv2.imshow('Intel RealSense - Max Res & Filters', display_image)

        # ตรวจสอบการกดปุ่ม 'q' หรือ 'ESC' (27)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:
            break

finally:
    # คืนทรัพยากรกล้อง
    pipeline.stop()
    cv2.destroyAllWindows()
    cv2.waitKey(1) 
    print("ปิดการสตรีมและเคลียร์หน้าต่างเรียบร้อยแล้ว")

# 3_1_ShowDepth

In [ ]:
import pyrealsense2 as rs
import numpy as np
import cv2

# 1. สร้าง Pipeline และ Config
pipeline = rs.pipeline()
config = rs.config()

# ตั้งค่าความละเอียดสูงสุด (RGB 1080p, Depth 720p)
config.enable_stream(rs.stream.color, 1920, 1080, rs.format.bgr8, 30)
config.enable_stream(rs.stream.depth, 1280, 720, rs.format.z16, 30)

print("กำลังเชื่อมต่อกล้อง Intel RealSense...")
profile = pipeline.start(config)

# เปิด IR Projector เบื้องหลัง
depth_sensor = profile.get_device().first_depth_sensor()
if depth_sensor.supports(rs.option.emitter_enabled):
    depth_sensor.set_option(rs.option.emitter_enabled, 1) 

# ตัวจัดตำแหน่งภาพดึงภาพ Depth ไปหาภาพสี (Color)
align_to = rs.stream.color
align = rs.align(align_to)

# ตั้งค่า Filters และ Colorizer
spatial_filter = rs.spatial_filter()
temporal_filter = rs.temporal_filter()
hole_filling = rs.hole_filling_filter()
colorizer = rs.colorizer()

print("กำลังสตรีมภาพเปรียบเทียบ Before / After Alignment...")
print("⚠️ คลิกที่หน้าต่างวิดีโอ แล้วกดปุ่ม 'q' หรือ 'ESC' เพื่อปิดโปรแกรม")

try:
    while True:
        # รอรับชุดข้อมูลเฟรมดั้งเดิม
        frames = pipeline.wait_for_frames()

        # [ส่วนที่ 1] ดึงภาพ Depth ดั้งเดิม (Raw Depth ก่อนผ่านกระบวนการ Align)
        raw_depth_frame = frames.get_depth_frame()

        # [ส่วนที่ 2] นำเฟรมไปผ่านกระบวนการจัดตำแหน่ง (Alignment Process)
        aligned_frames = align.process(frames)
        color_frame = aligned_frames.get_color_frame()
        aligned_depth_frame = aligned_frames.get_depth_frame()

        if not color_frame or not aligned_depth_frame or not raw_depth_frame:
            continue

        # นำภาพ Aligned Depth ไปผ่าน Filters เพื่อให้ภาพเนียนขึ้น
        filtered_aligned_depth = spatial_filter.process(aligned_depth_frame)
        filtered_aligned_depth = temporal_filter.process(filtered_aligned_depth)
        filtered_aligned_depth = hole_filling.process(filtered_aligned_depth)

        # ---------------------------------------------------------
        # แปลงเป็น Numpy Array และใส่สีภาพความลึก
        # ---------------------------------------------------------
        color_image = np.asanyarray(color_frame.get_data())
        
        # ใส่สีให้ Raw Depth (ภาพนี้ยังอยู่ที่ขนาด 1280x720 และมุมมองเดิม)
        raw_depth_image = np.asanyarray(colorizer.colorize(raw_depth_frame).get_data())
        
        # ใส่สีให้ Aligned Depth (ภาพนี้ถูกแปลงเป็น 1920x1080 และปรับมุมมองแล้ว)
        aligned_depth_image = np.asanyarray(colorizer.colorize(filtered_aligned_depth).get_data())

        # สร้างภาพ Overlay
        overlay_image = cv2.addWeighted(color_image, 0.5, aligned_depth_image, 0.5, 0)

        # ---------------------------------------------------------
        # ปรับขนาดภาพเพื่อให้ต่อกันเป็นตารางได้
        # ---------------------------------------------------------
        # เนื่องจากภาพ Raw Depth ดั้งเดิมเป็น 720p แต่ภาพอื่นเป็น 1080p 
        # เราต้องขยายขนาดของ Raw Depth ให้สูงเท่าภาพอื่นก่อนนำมาสแตกกัน
        raw_depth_resized = cv2.resize(raw_depth_image, (color_image.shape[1], color_image.shape[0]))

        # ---------------------------------------------------------
        # ใส่ข้อความกำกับ (Labels)
        # ---------------------------------------------------------
        font = cv2.FONT_HERSHEY_SIMPLEX
        color_label = (255, 255, 255)
        thickness = 3 
        
        cv2.putText(color_image, "1. RGB (Reference)", (20, 50), font, 1.3, color_label, thickness)
        cv2.putText(raw_depth_resized, "2. Raw Depth (Before Align - Shipped Position)", (20, 50), font, 1.3, color_label, thickness)
        cv2.putText(aligned_depth_image, "3. Aligned Depth (After Align)", (20, 50), font, 1.3, color_label, thickness)
        cv2.putText(overlay_image, "4. Depth Overlay (Perfect Match)", (20, 50), font, 1.3, color_label, thickness)

        # ---------------------------------------------------------
        # จัดตารางหน้าจอแสดงผล 2x2
        # ---------------------------------------------------------
        top_row = np.hstack((color_image, raw_depth_resized))
        bottom_row = np.hstack((aligned_depth_image, overlay_image))
        images_grid = np.vstack((top_row, bottom_row))

        # ย่อขนาดตารางรวมลง 40% เพื่อให้แสดงผลบนหน้าจอคอมพิวเตอร์ได้โดยไม่ล้น
        display_image = cv2.resize(images_grid, (0, 0), fx=0.4, fy=0.4)

        cv2.imshow('Intel RealSense - Alignment Comparison (2x2)', display_image)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:
            break

finally:
    pipeline.stop()
    cv2.destroyAllWindows()
    cv2.waitKey(1) 
    print("ปิดการสตรีมและเคลียร์หน้าต่างเรียบร้อยแล้ว")

3_AligDepthOverlay

In [ ]:
import pyrealsense2 as rs
import numpy as np
import cv2

# 1. สร้าง Pipeline และ Config
pipeline = rs.pipeline()
config = rs.config()

# ตั้งค่าความละเอียดสูงสุด
config.enable_stream(rs.stream.color, 1920, 1080, rs.format.bgr8, 30)
config.enable_stream(rs.stream.depth, 1280, 720, rs.format.z16, 30)

print("กำลังเชื่อมต่อกล้อง Intel RealSense...")
profile = pipeline.start(config)

# เปิด IR Projector เบื้องหลังเพื่อให้ Depth แม่นยำ
depth_sensor = profile.get_device().first_depth_sensor()
if depth_sensor.supports(rs.option.emitter_enabled):
    depth_sensor.set_option(rs.option.emitter_enabled, 1) 

# ---------------------------------------------------------
# 2. ตั้งค่า Alignment และ Filters
# ---------------------------------------------------------
# สร้างออบเจกต์สำหรับดัดภาพ Depth ให้ตรงกับมุมมองของภาพสี (Color)
align_to = rs.stream.color
align = rs.align(align_to)

spatial_filter = rs.spatial_filter()
temporal_filter = rs.temporal_filter()
hole_filling = rs.hole_filling_filter()
colorizer = rs.colorizer()

print("กำลังสตรีมภาพ (RGB | Aligned Depth | Overlay)...")
print("⚠️ คลิกที่หน้าต่างวิดีโอ แล้วกดปุ่ม 'q' หรือ 'ESC' เพื่อปิดโปรแกรม")

try:
    while True:
        frames = pipeline.wait_for_frames()

        # ---------------------------------------------------------
        # 3. จัดตำแหน่งภาพ (Align) ก่อนดึงเฟรม
        # ---------------------------------------------------------
        # นำเฟรมทั้งหมดไปผ่านกระบวนการ Align ภาพ Depth จะถูกขยายและดัดให้เท่ากับ RGB (1080p) ทันที
        aligned_frames = align.process(frames)

        color_frame = aligned_frames.get_color_frame()
        aligned_depth_frame = aligned_frames.get_depth_frame()

        if not color_frame or not aligned_depth_frame:
            continue

        # ---------------------------------------------------------
        # 4. นำภาพ Depth ที่จัดตำแหน่งแล้วไปผ่าน Filters
        # ---------------------------------------------------------
        filtered_depth = spatial_filter.process(aligned_depth_frame)
        filtered_depth = temporal_filter.process(filtered_depth)
        filtered_depth = hole_filling.process(filtered_depth)

        # ---------------------------------------------------------
        # 5. แปลงเป็น Numpy Array
        # ---------------------------------------------------------
        color_image = np.asanyarray(color_frame.get_data())
        depth_color_image = np.asanyarray(colorizer.colorize(filtered_depth).get_data())

        # สร้างภาพส่วนที่ 3 (Overlay) โดยการนำภาพ RGB กับ Depth มาผสมกันอย่างละ 50%
        overlay_image = cv2.addWeighted(color_image, 0.5, depth_color_image, 0.5, 0)

        # ---------------------------------------------------------
        # ใส่ข้อความกำกับ (Labels)
        # ---------------------------------------------------------
        font = cv2.FONT_HERSHEY_SIMPLEX
        color_label = (255, 255, 255) # ตัวอักษรสีขาว
        thickness = 3 
        
        cv2.putText(color_image, "1. RGB", (20, 50), font, 1.5, color_label, thickness)
        cv2.putText(depth_color_image, "2. Aligned Depth", (20, 50), font, 1.5, color_label, thickness)
        cv2.putText(overlay_image, "3. Depth Overlay", (20, 50), font, 1.5, color_label, thickness)

        # ---------------------------------------------------------
        # นำภาพมาต่อกันแนวนอน 3 ภาพ (ขนาด 1920x1080 ต่อกัน 3 ภาพ)
        # ---------------------------------------------------------
        images_hstack = np.hstack((color_image, depth_color_image, overlay_image))

        # ย่อขนาดลงเหลือ 33% เพื่อให้ความกว้างรวมพอดีกับหน้าจอคอมพิวเตอร์ทั่วไป (ประมาณ 1900px)
        display_image = cv2.resize(images_hstack, (0, 0), fx=0.33, fy=0.33)

        cv2.imshow('Intel RealSense - Alignment & Overlay', display_image)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:
            break

finally:
    pipeline.stop()
    cv2.destroyAllWindows()
    cv2.waitKey(1) 
    print("ปิดการสตรีมและเคลียร์หน้าต่างเรียบร้อยแล้ว")

# 4_DepthMeasure

In [ ]:
import pyrealsense2 as rs
import numpy as np
import cv2

# =========================================================
# 1. ตัวแปร Global สำหรับควบคุมกล่องเมาส์ (Mouse Callback)
# =========================================================
box_size = 80       # ขนาดความกว้าง/ยาวของกล่อง (พิกเซล)
box_x, box_y = 640, 360  # ตำแหน่งเริ่มต้น (กึ่งกลางหน้าจอ 1280x720)
is_dragging = False # สถานะการคลิกค้าง

def mouse_handler(event, x, y, flags, param):
    global box_x, box_y, is_dragging
    
    # เมื่อคลิกเมาส์ซ้าย -> ย้ายกล่องมาที่เคอร์เซอร์และเริ่มการลาก
    if event == cv2.EVENT_LBUTTONDOWN:
        is_dragging = True
        box_x, box_y = x, y
        
    # เมื่อขยับเมาส์ขณะคลิกค้าง -> อัปเดตตำแหน่งกล่อง
    elif event == cv2.EVENT_MOUSEMOVE:
        if is_dragging:
            box_x, box_y = x, y
            
    # เมื่อปล่อยคลิกซ้าย -> หยุดการลาก
    elif event == cv2.EVENT_LBUTTONUP:
        is_dragging = False

# =========================================================
# 2. ตั้งค่าการเชื่อมต่อกล้อง RealSense
# =========================================================
pipeline = rs.pipeline()
config = rs.config()

# ตั้งค่าความละเอียดที่ 1280x720 ทั้งคู่ เพื่อให้ไม่ต้องย่อภาพและเมาส์ชี้ได้แม่นยำ
config.enable_stream(rs.stream.color, 1280, 720, rs.format.bgr8, 30)
config.enable_stream(rs.stream.depth, 1280, 720, rs.format.z16, 30)

print("กำลังเชื่อมต่อกล้อง Intel RealSense...")
profile = pipeline.start(config)

# ดึงค่า Depth Scale ของกล้อง (ค่าตัวคูณเพื่อแปลงข้อมูลดิบเป็นหน่วย 'เมตร')
depth_sensor = profile.get_device().first_depth_sensor()
depth_scale = depth_sensor.get_depth_scale()

# เปิด IR Projector เบื้องหลัง
if depth_sensor.supports(rs.option.emitter_enabled):
    depth_sensor.set_option(rs.option.emitter_enabled, 1) 

# ตั้งค่า Alignment และ Filters
align = rs.align(rs.stream.color)
spatial_filter = rs.spatial_filter()
temporal_filter = rs.temporal_filter()
hole_filling = rs.hole_filling_filter()
colorizer = rs.colorizer()

# =========================================================
# 3. เตรียมหน้าต่างแสดงผลและผูกเมาส์เข้ากับหน้าต่าง
# =========================================================
window_name = 'Depth Overlay & Measurement (Click & Drag)'
cv2.namedWindow(window_name, cv2.WINDOW_AUTOSIZE)
cv2.setMouseCallback(window_name, mouse_handler)

print("กำลังสตรีมภาพ...")
print("🎯 คุณสามารถ 'คลิกและลากเมาส์' บนหน้าต่างเพื่อเลื่อนกล่องวัดระยะได้")
print("⚠️ กดปุ่ม 'q' หรือ 'ESC' เพื่อปิดโปรแกรม")

try:
    while True:
        frames = pipeline.wait_for_frames()

        # จัดตำแหน่ง (Align) ให้พิกเซลตรงกัน
        aligned_frames = align.process(frames)
        color_frame = aligned_frames.get_color_frame()
        aligned_depth_frame = aligned_frames.get_depth_frame()

        if not color_frame or not aligned_depth_frame:
            continue

        # นำภาพ Depth ผ่าน Filters เพื่อความเนียน
        filtered_depth = spatial_filter.process(aligned_depth_frame)
        filtered_depth = temporal_filter.process(filtered_depth)
        filtered_depth = hole_filling.process(filtered_depth)

        # แปลงข้อมูลเป็น Numpy Array
        color_image = np.asanyarray(color_frame.get_data())
        depth_data_array = np.asanyarray(filtered_depth.get_data()) # ค่าความลึกดิบสำหรับคำนวณ
        depth_color_image = np.asanyarray(colorizer.colorize(filtered_depth).get_data()) # ภาพสีความลึกสำหรับโชว์

        # สร้างภาพ Overlay (ผสม RGB 50% + Depth 50%)
        overlay_image = cv2.addWeighted(color_image, 0.5, depth_color_image, 0.5, 0)

        # =========================================================
        # 4. คำนวณระยะทางภายในกล่อง (ROI Distance Calculation)
        # =========================================================
        # ป้องกันไม่ให้ขอบเขตกล่องทะลุออกนอกภาพ (Clamp boundaries)
        y1 = max(0, box_y - box_size // 2)
        y2 = min(720, box_y + box_size // 2)
        x1 = max(0, box_x - box_size // 2)
        x2 = min(1280, box_x + box_size // 2)

        # ดึงชุดข้อมูลความลึกเฉพาะพื้นที่ในกล่อง (ROI)
        roi_depth = depth_data_array[y1:y2, x1:x2]

        # แปลงข้อมูลดิบเป็นหน่วย 'เมตร'
        distances = roi_depth * depth_scale

        # กรองเอาเฉพาะค่าที่มีความลึก (มากกว่า 0) เพื่อนำมาหาค่าเฉลี่ย
        valid_distances = distances[distances > 0]
        
        if len(valid_distances) > 0:
            avg_distance = np.mean(valid_distances)
            text_distance = f"Dist: {avg_distance:.3f} m"
            color_box = (0, 255, 0) # สีเขียวเมื่อจับระยะได้
        else:
            text_distance = "Dist: N/A"
            color_box = (0, 0, 255) # สีแดงเมื่อเป็นจุดบอด (หาความลึกไม่ได้)

        # =========================================================
        # 5. วาดกล่องและตัวหนังสือลงบนภาพ
        # =========================================================
        # วาดกรอบสี่เหลี่ยม
        cv2.rectangle(overlay_image, (x1, y1), (x2, y2), color_box, 2)
        
        # วาดจุดกึ่งกลาง (Center dot)
        cv2.circle(overlay_image, (box_x, box_y), 4, color_box, -1)

        # ใส่ข้อความบอกระยะทางไว้บนกล่อง
        cv2.putText(overlay_image, text_distance, (x1, y1 - 10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color_box, 2)
        
        # ใส่ข้อความแนะนำวิธีใช้ที่มุมซ้ายบน
        cv2.putText(overlay_image, "Click & drag the box to measure distance", 
                    (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        # =========================================================
        # แสดงผลภาพ
        # =========================================================
        cv2.imshow(window_name, overlay_image)

        # ตรวจสอบการกดปุ่ม 'q' หรือ 'ESC' (27)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:
            break

finally:
    # คืนทรัพยากรกล้อง
    pipeline.stop()
    cv2.destroyAllWindows()
    cv2.waitKey(1) 
    print("ปิดการสตรีมและเคลียร์หน้าต่างเรียบร้อยแล้ว")

# 5_DepthClipping

In [2]:
import pyrealsense2 as rs
import numpy as np
import cv2

# =========================================================
# 1. ตั้งค่าการเชื่อมต่อกล้อง
# =========================================================
pipeline = rs.pipeline()
config = rs.config()

# ใช้ความละเอียด 1280x720 เพื่อให้สัดส่วนภาพเท่ากันและทำงานได้ลื่นไหล
config.enable_stream(rs.stream.color, 1280, 720, rs.format.bgr8, 30)
config.enable_stream(rs.stream.depth, 1280, 720, rs.format.z16, 30)

print("กำลังเชื่อมต่อกล้อง Intel RealSense...")
profile = pipeline.start(config)

# ดึงค่า Depth Scale (ตัวคูณเพื่อแปลงค่าดิบให้เป็นหน่วยเมตร)
depth_sensor = profile.get_device().first_depth_sensor()
depth_scale = depth_sensor.get_depth_scale()

# เปิด IR Projector เบื้องหลัง
if depth_sensor.supports(rs.option.emitter_enabled):
    depth_sensor.set_option(rs.option.emitter_enabled, 0) 

# ตั้งค่า Alignment และ ตัวใส่สี
align = rs.align(rs.stream.color)
colorizer = rs.colorizer()

# =========================================================
# 2. ตั้งค่าระยะทางที่ต้องการตัด (Depth Clipping)
# =========================================================
clipping_distance_in_meters = 3.5 # กำหนดระยะที่ต้องการถมดำ (เมตร)

# แปลงหน่วย 'เมตร' ให้กลับเป็น 'ค่าดิบ (Raw Depth)' ที่กล้องเข้าใจ เพื่อให้ประมวลผลได้ไวขึ้น
clipping_distance = clipping_distance_in_meters / depth_scale

print(f"กำลังสตรีมภาพ (ตั้งค่าตัดฉากหลังที่ระยะ > {clipping_distance_in_meters} เมตร)...")
print("⚠️ คลิกที่หน้าต่างวิดีโอ แล้วกดปุ่ม 'q' หรือ 'ESC' เพื่อปิดโปรแกรม")

try:
    while True:
        frames = pipeline.wait_for_frames()

        # จัดตำแหน่ง (Align) ให้พิกเซลความลึกและภาพสีตรงกันเป๊ะ
        aligned_frames = align.process(frames)
        color_frame = aligned_frames.get_color_frame()
        aligned_depth_frame = aligned_frames.get_depth_frame()

        if not color_frame or not aligned_depth_frame:
            continue

        # =========================================================
        # 3. แปลงเป็น Numpy Array
        # =========================================================
        color_image = np.asanyarray(color_frame.get_data())
        depth_data = np.asanyarray(aligned_depth_frame.get_data()) # ค่าความลึกแบบดิบ
        depth_color_image = np.asanyarray(colorizer.colorize(aligned_depth_frame).get_data()) # ความลึกแบบมีสี

        # =========================================================
        # 4. ประมวลผลภาพทั้ง 3 ส่วน
        # =========================================================
        
        # [ส่วนที่ 1] ภาพ RGB ดั้งเดิม (มีอยู่ในตัวแปร color_image แล้ว)
        
        # [ส่วนที่ 2] ภาพ Overlay (RGB ผสมกับ Depth)
        overlay_image = cv2.addWeighted(color_image, 0.5, depth_color_image, 0.5, 0)
        
        # [ส่วนที่ 3] ภาพที่ถมดำเมื่อเกิน 3.5 เมตร (Background Removed)
        # เนื่องจากภาพสีมี 3 ช่องสี (BGR) แต่ภาพ Depth มีแค่ 1 มิติ
        # เราจึงต้องก็อปปี้มิติของ Depth ให้ซ้อนกัน 3 ชั้นก่อน (เพื่อใช้เป็นเงื่อนไข)
        depth_image_3d = np.dstack((depth_data, depth_data, depth_data)) 

        # สร้างภาพใหม่ โดยบอกว่า: 
        # ถ้าค่าความลึก > 3.5 เมตร หรือ หาค่าไม่ได้ (<= 0) ให้แทนที่ด้วยสีดำ (0)
        # ถ้าระยะไม่เกิน 3.5 เมตร ให้ใช้พิกเซลสีจาก color_image เหมือนเดิม
        bg_removed_image = np.where((depth_image_3d > clipping_distance) | (depth_image_3d <= 0), 0, color_image)

        # =========================================================
        # 5. ใส่ข้อความกำกับ (Labels)
        # =========================================================
        font = cv2.FONT_HERSHEY_SIMPLEX
        cv2.putText(color_image, "1. RGB", (20, 50), font, 1.2, (255, 255, 255), 2)
        cv2.putText(overlay_image, "2. Overlay", (20, 50), font, 1.2, (255, 255, 255), 2)
        cv2.putText(bg_removed_image, f"3. Clipped (< {clipping_distance_in_meters}m)", (20, 50), font, 1.2, (255, 255, 255), 2)

        # =========================================================
        # 6. จัด Layout และแสดงผล
        # =========================================================
        # นำภาพมาต่อกันแนวนอน 3 ภาพ
        images_hstack = np.hstack((color_image, overlay_image, bg_removed_image))

        # ย่อขนาดลง 40% (เหลือ 60%) เพื่อไม่ให้ล้นหน้าจอ (จาก 3840px จะเหลือประมาณ 1500px)
        display_image = cv2.resize(images_hstack, (0, 0), fx=0.4, fy=0.4)

        cv2.imshow('Intel RealSense - 3 Layers (Clipping)', display_image)

        # กด 'q' เพื่อออก
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:
            break

finally:
    pipeline.stop()
    cv2.destroyAllWindows()
    cv2.waitKey(1) 
    print("ปิดการสตรีมและเคลียร์หน้าต่างเรียบร้อยแล้ว")

กำลังเชื่อมต่อกล้อง Intel RealSense...
กำลังสตรีมภาพ (ตั้งค่าตัดฉากหลังที่ระยะ > 3.5 เมตร)...
⚠️ คลิกที่หน้าต่างวิดีโอ แล้วกดปุ่ม 'q' หรือ 'ESC' เพื่อปิดโปรแกรม
ปิดการสตรีมและเคลียร์หน้าต่างเรียบร้อยแล้ว


# 6_Post-Processing Filters + Clipping 
## High-Res + Alignment + Filters + Depth Clipping

In [4]:
import pyrealsense2 as rs
import numpy as np
import cv2

# =========================================================
# 1. ตั้งค่าการเชื่อมต่อกล้อง
# =========================================================
pipeline = rs.pipeline()
config = rs.config()

# ใช้ความละเอียด 1280x720 ทั้ง RGB และ Depth 
config.enable_stream(rs.stream.color, 1280, 720, rs.format.bgr8, 30)
config.enable_stream(rs.stream.depth, 1280, 720, rs.format.z16, 30)

print("กำลังเชื่อมต่อกล้อง Intel RealSense...")
profile = pipeline.start(config)

# ดึงค่า Depth Scale
depth_sensor = profile.get_device().first_depth_sensor()
depth_scale = depth_sensor.get_depth_scale()

# เปิด IR Projector เบื้องหลัง
if depth_sensor.supports(rs.option.emitter_enabled):
    depth_sensor.set_option(rs.option.emitter_enabled, 1) 

# =========================================================
# 2. ประกาศเครื่องมือสำหรับประมวลผลภาพ (Align & Filters)
# =========================================================
align = rs.align(rs.stream.color)
colorizer = rs.colorizer()

# ประกาศ Post-Processing Filters ทั้ง 3 ตัว
spatial_filter = rs.spatial_filter()
temporal_filter = rs.temporal_filter()
hole_filling = rs.hole_filling_filter()

# =========================================================
# 3. ตั้งค่าระยะตัดฉากหลัง (Clipping Distance)
# =========================================================
clipping_distance_in_meters = 3.5
clipping_distance = clipping_distance_in_meters / depth_scale

print(f"กำลังสตรีมภาพ (เพิ่ม Filters + ตัดฉากหลัง > {clipping_distance_in_meters}m)...")
print("⚠️ คลิกที่หน้าต่างวิดีโอ แล้วกดปุ่ม 'q' หรือ 'ESC' เพื่อปิดโปรแกรม")

try:
    while True:
        frames = pipeline.wait_for_frames()

        # [ขั้นตอนที่ 1] จัดตำแหน่งภาพ (Alignment) ให้พิกเซลตรงกันก่อน
        aligned_frames = align.process(frames)
        color_frame = aligned_frames.get_color_frame()
        aligned_depth_frame = aligned_frames.get_depth_frame()

        if not color_frame or not aligned_depth_frame:
            continue

        # [ขั้นตอนที่ 2] นำภาพ Depth ที่จัดตำแหน่งแล้ว ไปผ่าน Filters ให้ออกมาเนียนกริบ
        filtered_depth = spatial_filter.process(aligned_depth_frame)
        filtered_depth = temporal_filter.process(filtered_depth)
        filtered_depth = hole_filling.process(filtered_depth)

        # =========================================================
        # 4. แปลงเป็น Numpy Array
        # =========================================================
        color_image = np.asanyarray(color_frame.get_data())
        
        # ⚠️ สำคัญมาก: ต้องดึงข้อมูลดิบจาก filtered_depth (ภาพที่ผ่านฟิลเตอร์แล้ว)
        depth_data = np.asanyarray(filtered_depth.get_data()) 
        depth_color_image = np.asanyarray(colorizer.colorize(filtered_depth).get_data())

        # =========================================================
        # 5. ประมวลผลภาพทั้ง 3 ส่วน
        # =========================================================
        # [ส่วนที่ 1] RGB (color_image)
        
        # [ส่วนที่ 2] Overlay
        overlay_image = cv2.addWeighted(color_image, 0.5, depth_color_image, 0.5, 0)
        
        # [ส่วนที่ 3] Depth Clipping (ใช้ depth_data ที่ผ่าน Filter แล้วมาคำนวณ)
        depth_image_3d = np.dstack((depth_data, depth_data, depth_data)) 
        bg_removed_image = np.where((depth_image_3d > clipping_distance) | (depth_image_3d <= 0), 0, color_image)

        # =========================================================
        # 6. ใส่ข้อความและจัด Layout
        # =========================================================
        font = cv2.FONT_HERSHEY_SIMPLEX
        cv2.putText(color_image, "1. RGB", (20, 50), font, 1.2, (255, 255, 255), 2)
        cv2.putText(overlay_image, "2. Filtered Overlay", (20, 50), font, 1.2, (255, 255, 255), 2)
        cv2.putText(bg_removed_image, f"3. Clipped (< {clipping_distance_in_meters}m)", (20, 50), font, 1.2, (255, 255, 255), 2)

        images_hstack = np.hstack((color_image, overlay_image, bg_removed_image))
        display_image = cv2.resize(images_hstack, (0, 0), fx=0.4, fy=0.4)

        cv2.imshow('Intel RealSense - Pro Pipeline', display_image)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:
            break

finally:
    pipeline.stop()
    cv2.destroyAllWindows()
    cv2.waitKey(1) 
    print("ปิดการสตรีมเรียบร้อยแล้ว")

กำลังเชื่อมต่อกล้อง Intel RealSense...
กำลังสตรีมภาพ (เพิ่ม Filters + ตัดฉากหลัง > 3.5m)...
⚠️ คลิกที่หน้าต่างวิดีโอ แล้วกดปุ่ม 'q' หรือ 'ESC' เพื่อปิดโปรแกรม
ปิดการสตรีมเรียบร้อยแล้ว


# 7_Dynamic Depth-Color Mapping

In [6]:
import pyrealsense2 as rs
import numpy as np
import cv2

# =========================================================
# 1. ฟังก์ชันตัวจัดการ Trackbar
# =========================================================
def on_trackbar(val):
    # ฟังก์ชันเปล่าๆ เอาไว้รับค่าจาก Trackbar ของ OpenCV
    pass

# =========================================================
# 2. ตั้งค่ากล้อง
# =========================================================
pipeline = rs.pipeline()
config = rs.config()

# ใช้ความละเอียด 1280x720 
config.enable_stream(rs.stream.color, 1280, 720, rs.format.bgr8, 30)
config.enable_stream(rs.stream.depth, 1280, 720, rs.format.z16, 30)

print("กำลังเชื่อมต่อกล้อง...")
profile = pipeline.start(config)

# เปิด IR Projector เบื้องหลัง
depth_sensor = profile.get_device().first_depth_sensor()
if depth_sensor.supports(rs.option.emitter_enabled):
    depth_sensor.set_option(rs.option.emitter_enabled, 1)

# เครื่องมือจัดตำแหน่งและใส่สี
align = rs.align(rs.stream.color)
colorizer = rs.colorizer()

# =========================================================
# 3. เตรียมหน้าต่างและ Trackbar
# =========================================================
window_name = 'Universal Depth Overlay'
cv2.namedWindow(window_name, cv2.WINDOW_AUTOSIZE)

# สร้าง Trackbar ชื่อ 'Depth Opacity' ปรับค่าได้ 0-100 (เริ่มต้นที่ 50%)
cv2.createTrackbar('Depth Opacity (%)', window_name, 50, 100, on_trackbar)

print("กำลังสตรีมภาพ: เคลือบสีความลึกทับทุกวัตถุ")
print("🎯 ลองเลื่อนแถบ 'Depth Opacity' ด้านบนหน้าต่างเพื่อปรับความเข้ม")

try:
    while True:
        frames = pipeline.wait_for_frames()
        
        # จัดพิกเซลให้ตรงกัน (Align)
        aligned_frames = align.process(frames)
        color_frame = aligned_frames.get_color_frame()
        aligned_depth_frame = aligned_frames.get_depth_frame()

        if not color_frame or not aligned_depth_frame:
            continue

        # =========================================================
        # 4. แปลงข้อมูล
        # =========================================================
        color_image = np.asanyarray(color_frame.get_data())
        # ภาพความลึกที่ถูกระบายสีรุ้งแล้ว
        depth_color_image = np.asanyarray(colorizer.colorize(aligned_depth_frame).get_data())

        # =========================================================
        # 5. ดึงค่าจาก Trackbar มาผสมภาพ (Blending)
        # =========================================================
        # ดึงค่าปัจจุบัน (0-100) แล้วหาร 100 ให้อยู่ในรูปทศนิยม (0.0 - 1.0)
        depth_weight = cv2.getTrackbarPos('Depth Opacity (%)', window_name) / 100.0
        
        # น้ำหนักของภาพ RGB คือส่วนที่เหลือ (เช่น ถ้า Depth = 0.6, RGB = 0.4)
        rgb_weight = 1.0 - depth_weight

        # นำมาผสมกันด้วยสมการ: Output = (RGB * rgb_weight) + (Depth * depth_weight)
        overlay_image = cv2.addWeighted(color_image, rgb_weight, depth_color_image, depth_weight, 0)

        # =========================================================
        # 6. แสดงผล
        # =========================================================
        cv2.putText(overlay_image, f"RGB: {int(rgb_weight*100)}% | Depth: {int(depth_weight*100)}%", 
                    (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)

        # ย่อขนาดลง 40% เพื่อให้แสดงบนจอได้พอดี (ถ้าจอเล็กไป ปรับ fx, fy ได้ครับ)
        display_image = cv2.resize(overlay_image, (0, 0), fx=0.6, fy=0.6)

        cv2.imshow(window_name, display_image)

        if cv2.waitKey(1) & 0xFF == 27: # กด ESC เพื่อออก
            break

finally:
    pipeline.stop()
    cv2.destroyAllWindows()
    cv2.waitKey(1)

กำลังเชื่อมต่อกล้อง...
กำลังสตรีมภาพ: เคลือบสีความลึกทับทุกวัตถุ
🎯 ลองเลื่อนแถบ 'Depth Opacity' ด้านบนหน้าต่างเพื่อปรับความเข้ม


error: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\highgui\src\window_w32.cpp:2570: error: (-27:Null pointer) NULL window: 'Universal Depth Overlay' in function 'cvGetTrackbarPos'


# 8 EdgeDetection + MorphologicalOperations - RealTime

In [7]:
import pyrealsense2 as rs
import numpy as np
import cv2

# =========================================================
# 1. ตั้งค่ากล้อง RealSense
# =========================================================
pipeline = rs.pipeline()
config = rs.config()

config.enable_stream(rs.stream.color, 1280, 720, rs.format.bgr8, 30)
config.enable_stream(rs.stream.depth, 1280, 720, rs.format.z16, 30)

print("กำลังเชื่อมต่อกล้อง...")
profile = pipeline.start(config)

# เปิด IR Projector เบื้องหลัง
depth_sensor = profile.get_device().first_depth_sensor()
if depth_sensor.supports(rs.option.emitter_enabled):
    depth_sensor.set_option(rs.option.emitter_enabled, 1)

# เตรียมเครื่องมือ Alignment และ Filters
align = rs.align(rs.stream.color)
colorizer = rs.colorizer()

spatial = rs.spatial_filter()
temporal = rs.temporal_filter()
hole_filling = rs.hole_filling_filter()

print("กำลังสตรีมภาพ: ตีกรอบสีแดงรอบวัตถุจากแผนที่ความลึก (Depth Contours)")
print("⚠️ กด 'q' หรือ 'ESC' เพื่อปิดโปรแกรม")

try:
    while True:
        frames = pipeline.wait_for_frames()
        aligned_frames = align.process(frames)
        
        color_frame = aligned_frames.get_color_frame()
        depth_frame = aligned_frames.get_depth_frame()
        
        if not color_frame or not depth_frame:
            continue
            
        # ทำฟิลเตอร์ให้ภาพเนียนขึ้น
        filtered_depth = spatial.process(depth_frame)
        filtered_depth = temporal.process(filtered_depth)
        filtered_depth = hole_filling.process(filtered_depth)

        # แปลงเป็น Numpy Array
        color_image = np.asanyarray(color_frame.get_data())
        depth_color_image = np.asanyarray(colorizer.colorize(filtered_depth).get_data())
        
        # =========================================================
        # 2. กระบวนการหาขอบเขตวัตถุจากภาพความลึก (Depth Segmentation)
        # =========================================================
        # 2.1 แปลงภาพความลึกสีรุ้งให้เป็นภาพขาวดำ (Grayscale)
        gray_depth = cv2.cvtColor(depth_color_image, cv2.COLOR_BGR2GRAY)
        
        # 2.2 เบลอภาพเล็กน้อยเพื่อลด Noise
        blurred = cv2.GaussianBlur(gray_depth, (5, 5), 0)
        
        # 2.3 หาขอบของวัตถุ (Edge Detection)
        edges = cv2.Canny(blurred, 30, 100)
        
        # 2.4 ใช้ Morphological Dilation (การขยายพิกเซล) 
        # เพื่อเชื่อมเส้นขอบที่ขาดให้ติดกันเป็นกลุ่มก้อนเดียวกัน
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
        dilated = cv2.dilate(edges, kernel, iterations=2)
        
        # 2.5 หาเส้นรอบรูป (Contours) จากภาพที่เชื่อมขอบแล้ว
        contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        # =========================================================
        # 3. วาดกรอบสีแดงทับลงไป
        # =========================================================
        # ใช้ภาพผสมระหว่าง RGB (30%) และ Depth (70%) เพื่อให้ดูคล้ายภาพที่คุณส่งมา
        output_image = cv2.addWeighted(color_image, 0.3, depth_color_image, 0.7, 0)
        
        for contour in contours:
            # คำนวณขนาดพื้นที่ของ Contours
            area = cv2.contourArea(contour)
            
            # กรองเอาเฉพาะวัตถุที่มีขนาดใหญ่พอ (ป้องกันการตีกรอบจุด Noise เล็กๆ)
            if area > 4000: 
                # หาพิกัดสี่เหลี่ยมที่ครอบพื้นที่นั้น
                x, y, w, h = cv2.boundingRect(contour)
                
                # วาดกรอบสีแดง (B=0, G=0, R=255) ความหนา 4 พิกเซล
                cv2.rectangle(output_image, (x, y), (x + w, y + h), (0, 0, 255), 4)
                
        # =========================================================
        # 4. แสดงผล
        # =========================================================
        # ย่อขนาดลง 40% เพื่อให้แสดงผลบนหน้าจอได้พอดี
        display_image = cv2.resize(output_image, (0, 0), fx=0.6, fy=0.6)
        
        cv2.imshow('Depth Bounding Boxes', display_image)
        
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:
            break

finally:
    pipeline.stop()
    cv2.destroyAllWindows()
    cv2.waitKey(1)

กำลังเชื่อมต่อกล้อง...
กำลังสตรีมภาพ: ตีกรอบสีแดงรอบวัตถุจากแผนที่ความลึก (Depth Contours)
⚠️ กด 'q' หรือ 'ESC' เพื่อปิดโปรแกรม


# 9 from 8 add TemporalAccumulation (MotionHistory)
การเก็บภาพ 30 เฟรม/วินาที เป็นเวลา 10 วินาที เท่ากับเราต้องประมวลผลถึง 300 เฟรมตลอดเวลา หากเราใช้ลูปบวกภาพทั้ง 300 เฟรมใหม่ทุกรอบ คอมพิวเตอร์จะกระตุกแน่นอน
จึงใช้ "Running Sum (การบวกสะสม)" คือเมื่อมีเฟรมใหม่เข้ามา เราจะเอาไปบวกเพิ่ม และลบเฟรมที่เก่าที่สุด (เฟรมที่ 301) ทิ้งไป ทำให้โค้ดทำงานได้เร็วระดับเสี้ยววินาที $O(1)$ ครับ

In [9]:
import pyrealsense2 as rs
import numpy as np
import cv2
from collections import deque

# =========================================================
# 1. ตั้งค่ากล้อง
# =========================================================
pipeline = rs.pipeline()
config = rs.config()

# ใช้ความละเอียด 1280x720 ที่ 30 FPS
FPS = 30
config.enable_stream(rs.stream.color, 1280, 720, rs.format.bgr8, FPS)
config.enable_stream(rs.stream.depth, 1280, 720, rs.format.z16, FPS)

print("กำลังเชื่อมต่อกล้อง...")
profile = pipeline.start(config)

depth_sensor = profile.get_device().first_depth_sensor()
depth_scale = depth_sensor.get_depth_scale()
if depth_sensor.supports(rs.option.emitter_enabled):
    depth_sensor.set_option(rs.option.emitter_enabled, 1)

align = rs.align(rs.stream.color)
colorizer = rs.colorizer()

spatial = rs.spatial_filter()
hole_filling = rs.hole_filling_filter()

# =========================================================
# 2. ตั้งค่าคิวเก็บข้อมูล 10 วินาที
# =========================================================
SECONDS = 10
MAX_FRAMES = FPS * SECONDS

mask_queue = deque(maxlen=MAX_FRAMES)
running_sum = np.zeros((720, 1280), dtype=np.int32)

# ระยะเป้าหมาย (เมตร)
MIN_DIST = 0.1
MAX_DIST = 1.5

print(f"กำลังสตรีมภาพ (2x2 Grid): เก็บข้อมูลสะสม {SECONDS} วินาที")
print("⚠️ กด 'q' หรือ 'ESC' เพื่อปิดโปรแกรม")

try:
    while True:
        frames = pipeline.wait_for_frames()
        aligned_frames = align.process(frames)
        
        color_frame = aligned_frames.get_color_frame()
        depth_frame = aligned_frames.get_depth_frame()
        
        if not color_frame or not depth_frame:
            continue
            
        filtered_depth = spatial.process(depth_frame)
        filtered_depth = hole_filling.process(filtered_depth)

        # ดึงข้อมูลเป็น Numpy
        color_image = np.asanyarray(color_frame.get_data())
        depth_data = np.asanyarray(filtered_depth.get_data()) * depth_scale
        depth_color_image = np.asanyarray(colorizer.colorize(filtered_depth).get_data())

        # =========================================================
        # 3. สร้าง Mask ปัจจุบัน และสะสม 10 วินาที
        # =========================================================
        current_mask = np.where((depth_data > MIN_DIST) & (depth_data < MAX_DIST), 1, 0).astype(np.uint8)
        
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
        current_mask = cv2.morphologyEx(current_mask, cv2.MORPH_OPEN, kernel)

        if len(mask_queue) == MAX_FRAMES:
            oldest_mask = mask_queue.popleft()
            running_sum -= oldest_mask
            
        mask_queue.append(current_mask)
        running_sum += current_mask

        accumulated_mask_255 = np.where(running_sum > 0, 255, 0).astype(np.uint8)
        
        # =========================================================
        # 4. เตรียมภาพทั้ง 4 ส่วน (2x2 Grid)
        # =========================================================
        # 1. บนซ้าย: RGB Real-time
        top_left = color_image.copy()
        
        # 2. บนขวา: RGB Accumulated (ดรอปแสง + ถมสีเหลือง)
        top_right = cv2.addWeighted(color_image, 0.5, np.zeros_like(color_image), 0, 0)
        yellow_trail = np.zeros_like(color_image)
        yellow_trail[accumulated_mask_255 == 255] = [0, 255, 255]
        top_right = cv2.addWeighted(top_right, 1.0, yellow_trail, 0.4, 0)

        # 3. ล่างซ้าย: Depth Color Real-time
        bottom_left = depth_color_image.copy()

        # 4. ล่างขวา: Accumulated Binary Mask (แปลงจาก 1 ช่องสี เป็น 3 ช่องสี เพื่อให้วาดกรอบแดงได้)
        bottom_right = cv2.cvtColor(accumulated_mask_255, cv2.COLOR_GRAY2BGR)

        # =========================================================
        # 5. วาดกรอบสี่เหลี่ยม
        # =========================================================
        # --- วาด Real-time (ลงบนกรอบ 1 และ 3) ---
        current_mask_255 = (current_mask * 255).astype(np.uint8)
        contours_current, _ = cv2.findContours(current_mask_255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours_current:
            if cv2.contourArea(cnt) > 2000:
                x, y, w, h = cv2.boundingRect(cnt)
                cv2.rectangle(top_left, (x, y), (x+w, y+h), (0, 255, 0), 3)
                cv2.rectangle(bottom_left, (x, y), (x+w, y+h), (0, 255, 0), 3)
                
        # --- วาด Accumulated (ลงบนกรอบ 2 และ 4) ---
        contours_acc, _ = cv2.findContours(accumulated_mask_255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours_acc:
            if cv2.contourArea(cnt) > 2000:
                x, y, w, h = cv2.boundingRect(cnt)
                cv2.rectangle(top_right, (x, y), (x+w, y+h), (0, 0, 255), 4)
                cv2.rectangle(bottom_right, (x, y), (x+w, y+h), (0, 0, 255), 4)

        # =========================================================
        # 6. ใส่ตัวหนังสือและประกอบร่าง
        # =========================================================
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale, thick, color_txt = 1.2, 3, (255, 255, 255)

        cv2.putText(top_left, "1. RGB (Real-time)", (20, 50), font, font_scale, color_txt, thick)
        
        current_sec = len(mask_queue) / FPS
        cv2.putText(top_right, f"2. RGB (Accumulated {current_sec:.1f}s)", (20, 50), font, font_scale, color_txt, thick)
        
        cv2.putText(bottom_left, "3. Depth (Real-time)", (20, 50), font, font_scale, color_txt, thick)
        cv2.putText(bottom_right, "4. Mask (Logic Behind Accumulated)", (20, 50), font, font_scale, color_txt, thick)

        # ประกอบร่าง (Vstack / Hstack)
        row1 = np.hstack((top_left, top_right))
        row2 = np.hstack((bottom_left, bottom_right))
        grid = np.vstack((row1, row2))

        # ภาพรวมมีขนาดใหญ่มาก (2560x1440) จึงต้องย่อลง 50% เพื่อให้โชว์ในจอพอดีที่ (1280x720)
        display_image = cv2.resize(grid, (0, 0), fx=0.5, fy=0.5)

        cv2.imshow('Depth Tracking 2x2', display_image)

        if cv2.waitKey(1) & 0xFF == 27:
            break

finally:
    pipeline.stop()
    cv2.destroyAllWindows()
    cv2.waitKey(1)

กำลังเชื่อมต่อกล้อง...
กำลังสตรีมภาพ (2x2 Grid): เก็บข้อมูลสะสม 10 วินาที
⚠️ กด 'q' หรือ 'ESC' เพื่อปิดโปรแกรม


KeyboardInterrupt: 

In [16]:
import pyrealsense2 as rs
import numpy as np
import cv2

# =========================================================
# 1. ตั้งค่ากล้อง RealSense
# =========================================================
pipeline = rs.pipeline()
config = rs.config()

# ใช้ความละเอียด 1280x720 ที่ 30 FPS ทั้งคู่
WIDTH, HEIGHT, FPS = 1280, 720, 30
config.enable_stream(rs.stream.color, WIDTH, HEIGHT, rs.format.bgr8, FPS)
config.enable_stream(rs.stream.depth, WIDTH, HEIGHT, rs.format.z16, FPS)

print("กำลังเชื่อมต่อกล้อง...")
profile = pipeline.start(config)

depth_sensor = profile.get_device().first_depth_sensor()
if depth_sensor.supports(rs.option.emitter_enabled):
    depth_sensor.set_option(rs.option.emitter_enabled, 1) # เปิด IR Projector

align = rs.align(rs.stream.color)
colorizer = rs.colorizer()

spatial = rs.spatial_filter()
hole_filling = rs.hole_filling_filter()

# =========================================================
# 2. กำหนดช่วงสี HSV สำหรับแยกความลึก (JET Colormap)
# =========================================================
# โครงสร้าง: "ชื่อกลุ่ม": [ค่า HSV ล่าง, ค่า HSV บน, สีวาดกรอบ BGR]
color_groups = {
    "Red (Very Close)": [np.array([0, 150, 50]), np.array([10, 255, 255]), (0, 0, 255)],
    "Orange (Close)": [np.array([11, 150, 50]), np.array([22, 255, 255]), (0, 165, 255)],
    "Yellow (Medium)": [np.array([23, 150, 50]), np.array([35, 255, 255]), (0, 255, 255)],
    "Green (Far)": [np.array([40, 100, 50]), np.array([85, 255, 255]), (0, 255, 0)],
    "Blue (Very Far)": [np.array([95, 150, 50]), np.array([130, 255, 255]), (255, 0, 0)]
}

# กำหนดสีเขียวสำหรับเขียนหัวข้อ quadrants
HEADER_COLOR = (0, 255, 0)
HEADER_FONT = cv2.FONT_HERSHEY_SIMPLEX
HEADER_SCALE = 1.0
HEADER_THICKNESS = 2

print("กำลังสตรีมภาพ (2x2 Grid): ตีกรอบวัตถุตามกลุ่มสีความลึก")
print("⚠️ กด 'q' หรือ 'ESC' เพื่อปิดโปรแกรม")

try:
    while True:
        # รอรับชุดข้อมูลเฟรมล่าสุด
        frames = pipeline.wait_for_frames()
        aligned_frames = align.process(frames)
        
        color_frame = aligned_frames.get_color_frame()
        depth_frame = aligned_frames.get_depth_frame()
        
        if not color_frame or not depth_frame:
            continue
            
        # กรองความลึกให้เนียน
        filtered_depth = spatial.process(depth_frame)
        filtered_depth = hole_filling.process(filtered_depth)

        # แปลงเป็น Numpy
        raw_color = np.asanyarray(color_frame.get_data())
        raw_depth_color = np.asanyarray(colorizer.colorize(filtered_depth).get_data())

        # =========================================================
        # 3. เตรียมภาพทั้ง 4 สำหรับ 2x2 Grid
        # =========================================================
        # เราเก็บภาพ RAW ไว้ และสร้างก๊อปปี้มาสำหรับวาดกรอบ
        processed_color = raw_color.copy()
        processed_depth = raw_depth_color.copy()

        # แปลงภาพ Depth Color เป็น HSV เพื่อหาขอบเขตตามช่วงสี
        hsv_depth = cv2.cvtColor(raw_depth_color, cv2.COLOR_BGR2HSV)

        # =========================================================
        # 4. วนลูปหากลุ่มสีความลึกและตีกรอบสี่เหลี่ยม (Drawing)
        # =========================================================
        for name, (lower, upper, draw_color) in color_groups.items():
            
            # สร้าง Mask หาช่วงสีนั้นๆ
            mask = cv2.inRange(hsv_depth, lower, upper)
            
            # พิเศษ: ดักจับสีแดงกรณีค่า Hue ไปตกที่ปลายขอบ (170-180)
            if "Red" in name:
                mask_red_wrap = cv2.inRange(hsv_depth, np.array([170, 150, 50]), np.array([180, 255, 255]))
                mask = cv2.bitwise_or(mask, mask_red_wrap)

            # ใช้ Morphology ลบ Noise (จุดสีเล็กๆ)
            kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (11, 11))
            mask_cleaned = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

            # หากรอบ (Contours)
            contours, _ = cv2.findContours(mask_cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            for cnt in contours:
                # กรองเอาเฉพาะกลุ่มสีที่มีขนาดใหญ่พอ (area > 3000px)
                if cv2.contourArea(cnt) > 3000:
                    x, y, w, h = cv2.boundingRect(cnt)
                    
                    # วาดกรอบสี่เหลี่ยมลงบนภาพ RGB และ Depth ฉบับก๊อปปี้
                    cv2.rectangle(processed_color, (x, y), (x+w, y+h), draw_color, 4)
                    cv2.rectangle(processed_depth, (x, y), (x+w, y+h), draw_color, 4)
                    
                    # ใส่ข้อความระยะสี
                    cv2.putText(processed_color, name.split()[0], (x, y-15), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, draw_color, 2)
                    cv2.putText(processed_depth, name.split()[0], (x, y-15), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, draw_color, 2)

        # =========================================================
        # 5. ใส่หัวข้อช่องการแสดงผล (Quadrants)
        # =========================================================
        # บนซ้าย (1)
        top_left = raw_color.copy()
        cv2.putText(top_left, "1. Color RAW", (20, 40), HEADER_FONT, HEADER_SCALE, HEADER_COLOR, HEADER_THICKNESS)
        
        # บนขวา (2)
        top_right = raw_depth_color.copy()
        cv2.putText(top_right, "2. Depth Colorizer RAW", (20, 40), HEADER_FONT, HEADER_SCALE, HEADER_COLOR, HEADER_THICKNESS)
        
        # ล่างซ้าย (3)
        bottom_left = processed_color
        cv2.putText(bottom_left, "3. Color (Segmentation)", (20, 40), HEADER_FONT, HEADER_SCALE, HEADER_COLOR, HEADER_THICKNESS)
        
        # ล่างขวา (4)
        bottom_right = processed_depth
        cv2.putText(bottom_right, "4. Depth Colorizer (Segmentation)", (20, 40), HEADER_FONT, HEADER_SCALE, HEADER_COLOR, HEADER_THICKNESS)

        # =========================================================
        # 6. ประกอบร่างภาพเป็น 2x2 Grid และย่อขนาดเพื่อแสดงผล
        # =========================================================
        # นำช่องซ้าย-ขวามาต่อกันแนวนอน (Hstack) เพื่อสร้างแถว
        row1 = np.hstack((top_left, top_right))
        row2 = np.hstack((bottom_left, bottom_right))
        
        # นำแถวบน-ล่างมาต่อกันแนวตั้ง (Vstack) เพื่อสร้าง Grid รวม
        grid = np.vstack((row1, row2))

        # ขนาดรวมของ Grid ตอนนี้คือ (1280x2, 720x2) = (2560, 1440) ซึ่งใหญ่กว่าจอทั่วไป
        # ย่อขนาดลง 50% ให้เหลือ (1280, 720) พิกเซลเพื่อไม่ให้ล้นหน้าจอ
        display_image = cv2.resize(grid, (0, 0), fx=0.5, fy=0.5)

        # แสดงหน้าต่าง
        cv2.imshow('Depth Tracking Dashboard (2x2 Color Segmentation)', display_image)

        # กด 'q' หรือ ESC เพื่อออก
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:
            break

finally:
    # คำสั่งปิดสตรีมกล้อง
    pipeline.stop()
    cv2.destroyAllWindows()
    cv2.waitKey(1)

กำลังเชื่อมต่อกล้อง...
กำลังสตรีมภาพ (2x2 Grid): ตีกรอบวัตถุตามกลุ่มสีความลึก
⚠️ กด 'q' หรือ 'ESC' เพื่อปิดโปรแกรม


KeyboardInterrupt: 

# 10 4x2

In [18]:
import pyrealsense2 as rs
import numpy as np
import cv2
from collections import deque

# =========================================================
# 1. ตั้งค่ากล้อง RealSense
# =========================================================
pipeline = rs.pipeline()
config = rs.config()

WIDTH, HEIGHT, FPS = 1280, 720, 30
config.enable_stream(rs.stream.color, WIDTH, HEIGHT, rs.format.bgr8, FPS)
config.enable_stream(rs.stream.depth, WIDTH, HEIGHT, rs.format.z16, FPS)

print("กำลังเชื่อมต่อกล้อง...")
profile = pipeline.start(config)

depth_sensor = profile.get_device().first_depth_sensor()
if depth_sensor.supports(rs.option.emitter_enabled):
    depth_sensor.set_option(rs.option.emitter_enabled, 1)

align = rs.align(rs.stream.color)
colorizer = rs.colorizer()
spatial = rs.spatial_filter()
hole_filling = rs.hole_filling_filter()

# =========================================================
# 2. ตั้งค่าการสะสมข้อมูล (Running Sum) แยกตามกลุ่มสี
# =========================================================
SECONDS = 10
MAX_FRAMES = FPS * SECONDS

color_groups = {
    "Red (Very Close)": [np.array([0, 150, 50]), np.array([10, 255, 255]), (0, 0, 255)],
    "Orange (Close)": [np.array([11, 150, 50]), np.array([22, 255, 255]), (0, 165, 255)],
    "Yellow (Medium)": [np.array([23, 150, 50]), np.array([35, 255, 255]), (0, 255, 255)],
    "Green (Far)": [np.array([40, 100, 50]), np.array([85, 255, 255]), (0, 255, 0)],
    "Blue (Very Far)": [np.array([95, 150, 50]), np.array([130, 255, 255]), (255, 0, 0)]
}

# คิวเก็บ Mask แต่ละเฟรม (เก็บรวมเป็น Dictionary)
mask_queue = deque(maxlen=MAX_FRAMES)

# สร้าง Running Sum แยกสำหรับแต่ละกลุ่มสี 5 ตัว
running_sums = {name: np.zeros((HEIGHT, WIDTH), dtype=np.int32) for name in color_groups.keys()}
# สร้าง Running Sum รวม (สำหรับแถวที่ 3)
general_running_sum = np.zeros((HEIGHT, WIDTH), dtype=np.int32)

print(f"กำลังสตรีมภาพ (ตาราง 4x2): เพิ่ม Segmentation บนข้อมูลสะสม {SECONDS} วินาที")
print("⚠️ กด 'q' หรือ 'ESC' เพื่อปิดโปรแกรม")

try:
    while True:
        frames = pipeline.wait_for_frames()
        aligned_frames = align.process(frames)
        
        color_frame = aligned_frames.get_color_frame()
        depth_frame = aligned_frames.get_depth_frame()
        
        if not color_frame or not depth_frame:
            continue
            
        filtered_depth = spatial.process(depth_frame)
        filtered_depth = hole_filling.process(filtered_depth)

        raw_color = np.asanyarray(color_frame.get_data())
        raw_depth_color = np.asanyarray(colorizer.colorize(filtered_depth).get_data())

        # =========================================================
        # แถวที่ 1: ภาพ RAW (ช่อง 1 และ 2)
        # =========================================================
        live_color_seg = raw_color.copy()
        live_depth_seg = raw_depth_color.copy()
        
        hsv_depth = cv2.cvtColor(raw_depth_color, cv2.COLOR_BGR2HSV)

        # Dictionary เก็บ Mask ประจำเฟรมนี้
        current_masks = {}
        current_general_mask = np.zeros((HEIGHT, WIDTH), dtype=np.uint8)

        # =========================================================
        # แถวที่ 2: Segmentation แบบ Real-time (ช่อง 3 และ 4)
        # =========================================================
        for name, (lower, upper, draw_color) in color_groups.items():
            mask_color = cv2.inRange(hsv_depth, lower, upper)
            if "Red" in name:
                mask_red_wrap = cv2.inRange(hsv_depth, np.array([170, 150, 50]), np.array([180, 255, 255]))
                mask_color = cv2.bitwise_or(mask_color, mask_red_wrap)
                
            kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (11, 11))
            mask_color_clean = cv2.morphologyEx(mask_color, cv2.MORPH_OPEN, kernel)
            
            # เก็บ Mask ของเฟรมนี้ลง Dictionary ไว้สะสม
            current_masks[name] = mask_color_clean
            # รวมเข้า Mask การขยับแบบรวม (General)
            current_general_mask = cv2.bitwise_or(current_general_mask, mask_color_clean)
            
            contours_rt, _ = cv2.findContours(mask_color_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            for cnt in contours_rt:
                if cv2.contourArea(cnt) > 3000:
                    x, y, w, h = cv2.boundingRect(cnt)
                    cv2.rectangle(live_color_seg, (x, y), (x+w, y+h), draw_color, 3)
                    cv2.rectangle(live_depth_seg, (x, y), (x+w, y+h), draw_color, 3)

        # =========================================================
        # ระบบสะสมข้อมูล (Update Running Sums)
        # =========================================================
        if len(mask_queue) == MAX_FRAMES:
            old_masks, old_general = mask_queue.popleft()
            for name in color_groups.keys():
                running_sums[name] -= old_masks[name]
            general_running_sum -= old_general
            
        mask_queue.append((current_masks, current_general_mask))
        for name in color_groups.keys():
            running_sums[name] += current_masks[name]
        general_running_sum += current_general_mask

        # =========================================================
        # แถวที่ 3: Running Sum แบบรวม (ช่อง 5 และ 6)
        # =========================================================
        gen_acc_mask_255 = np.where(general_running_sum > 0, 255, 0).astype(np.uint8)

        acc_gen_color = cv2.addWeighted(raw_color, 0.4, np.zeros_like(raw_color), 0, 0)
        acc_gen_depth = cv2.addWeighted(raw_depth_color, 0.4, np.zeros_like(raw_depth_color), 0, 0)
        
        trail_white = np.zeros_like(raw_color)
        trail_white[gen_acc_mask_255 == 255] = [255, 255, 255] # ถมรอยสีขาว
        
        acc_gen_color = cv2.addWeighted(acc_gen_color, 1.0, trail_white, 0.4, 0)
        acc_gen_depth = cv2.addWeighted(acc_gen_depth, 1.0, trail_white, 0.4, 0)

        contours_gen_acc, _ = cv2.findContours(gen_acc_mask_255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours_gen_acc:
            if cv2.contourArea(cnt) > 3000:
                x, y, w, h = cv2.boundingRect(cnt)
                cv2.rectangle(acc_gen_color, (x, y), (x+w, y+h), (255, 255, 255), 3)
                cv2.rectangle(acc_gen_depth, (x, y), (x+w, y+h), (255, 255, 255), 3)

        # =========================================================
        # แถวที่ 4: Segmentation บนข้อมูลสะสม (ช่อง 7 และ 8)
        # =========================================================
        # สร้าง Canvas สำหรับถมสีแยกตามกลุ่ม
        color_trails_canvas = np.zeros_like(raw_color)
        
        acc_seg_color = cv2.addWeighted(raw_color, 0.4, np.zeros_like(raw_color), 0, 0)
        acc_seg_depth = cv2.addWeighted(raw_depth_color, 0.4, np.zeros_like(raw_depth_color), 0, 0)

        # วนลูปอ่านข้อมูลสะสมแยกตามสี (Segmentation on History)
        for name, (_, _, draw_color) in color_groups.items():
            # ดึง Mask สะสมเฉพาะของกลุ่มสีนี้ออกมา
            group_acc_mask = np.where(running_sums[name] > 0, 255, 0).astype(np.uint8)
            
            # แต้มสีลงบน Canvas (เพื่อโชว์คราบสีของแต่ละระยะ)
            color_trails_canvas[group_acc_mask == 255] = draw_color
            
            # วาดกรอบสี่เหลี่ยมของกลุ่มสีนี้
            contours_grp_acc, _ = cv2.findContours(group_acc_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            for cnt in contours_grp_acc:
                if cv2.contourArea(cnt) > 3000:
                    x, y, w, h = cv2.boundingRect(cnt)
                    cv2.rectangle(acc_seg_color, (x, y), (x+w, y+h), draw_color, 4)
                    cv2.rectangle(acc_seg_depth, (x, y), (x+w, y+h), draw_color, 4)
        
        # ผสมภาพคราบสีเข้ากับภาพพื้นหลัง
        acc_seg_color = cv2.addWeighted(acc_seg_color, 1.0, color_trails_canvas, 0.6, 0)
        acc_seg_depth = cv2.addWeighted(acc_seg_depth, 1.0, color_trails_canvas, 0.6, 0)

        # =========================================================
        # การจัดตาราง 4x2 และใส่ข้อความ
        # =========================================================
        font = cv2.FONT_HERSHEY_SIMPLEX
        f_scale, thick, txt_col = 1.0, 2, (255, 255, 255)

        # แถว 1
        cv2.putText(raw_color, "1. Color RAW", (20, 40), font, f_scale, txt_col, thick)
        cv2.putText(raw_depth_color, "2. Depth Colorizer RAW", (20, 40), font, f_scale, txt_col, thick)
        row1 = np.hstack((raw_color, raw_depth_color))

        # แถว 2
        cv2.putText(live_color_seg, "3. Color (Live Seg)", (20, 40), font, f_scale, txt_col, thick)
        cv2.putText(live_depth_seg, "4. Depth Colorizer (Live Seg)", (20, 40), font, f_scale, txt_col, thick)
        row2 = np.hstack((live_color_seg, live_depth_seg))

        # แถว 3
        current_sec = len(mask_queue) / FPS
        cv2.putText(acc_gen_color, f"5. RunningSum Color ({current_sec:.1f}s)", (20, 40), font, f_scale, txt_col, thick)
        cv2.putText(acc_gen_depth, f"6. RunningSum Depth ({current_sec:.1f}s)", (20, 40), font, f_scale, txt_col, thick)
        row3 = np.hstack((acc_gen_color, acc_gen_depth))

        # แถว 4
        cv2.putText(acc_seg_color, f"7. Acc. Seg Color ({current_sec:.1f}s)", (20, 40), font, f_scale, txt_col, thick)
        cv2.putText(acc_seg_depth, f"8. Acc. Seg Depth ({current_sec:.1f}s)", (20, 40), font, f_scale, txt_col, thick)
        row4 = np.hstack((acc_seg_color, acc_seg_depth))

        # รวม 4 แถว
        grid_4x2 = np.vstack((row1, row2, row3, row4))

        # หน้าต่าง 4 แถวจะมีความสูงมาก (720x4 = 2880px) จึงต้องย่อลงมาที่ 35% เพื่อให้พอดีกับจอ
        display_image = cv2.resize(grid_4x2, (0, 0), fx=0.35, fy=0.35)

        # ตีเส้นแบ่งตาราง
        h_d, w_d, _ = display_image.shape
        cv2.line(display_image, (w_d//2, 0), (w_d//2, h_d), (50, 50, 50), 2) # เส้นแบ่งกลาง
        cv2.line(display_image, (0, h_d//4), (w_d, h_d//4), (50, 50, 50), 2) # เส้นนอน 1
        cv2.line(display_image, (0, 2*h_d//4), (w_d, 2*h_d//4), (50, 50, 50), 2) # เส้นนอน 2
        cv2.line(display_image, (0, 3*h_d//4), (w_d, 3*h_d//4), (50, 50, 50), 2) # เส้นนอน 3

        cv2.imshow('Intel RealSense - 4x2 Ultimate Analysis Dashboard', display_image)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:
            break

finally:
    pipeline.stop()
    cv2.destroyAllWindows()
    cv2.waitKey(1)

กำลังเชื่อมต่อกล้อง...
กำลังสตรีมภาพ (ตาราง 4x2): เพิ่ม Segmentation บนข้อมูลสะสม 10 วินาที
⚠️ กด 'q' หรือ 'ESC' เพื่อปิดโปรแกรม
